# End-to-End Machine Learning Workflow: Maternal Health Risk

Dataset: UCI Maternal Health Risk (ID 863).

In [17]:
%pip install -U ucimlrepo

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Manav\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [18]:
from ucimlrepo import fetch_ucirepo

maternal_health_risk = fetch_ucirepo(id=863)
X = maternal_health_risk.data.features.copy()
y = maternal_health_risk.data.targets.copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nX.head():")
display(X.head())
print("\ny.head():")
display(y.head())

Features shape: (1014, 6)
Target shape: (1014, 1)

X.head():


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
0,25,130,80,15.0,98.0,86
1,35,140,90,13.0,98.0,70
2,29,90,70,8.0,100.0,80
3,30,140,85,7.0,98.0,70
4,35,120,60,6.1,98.0,76



y.head():


,RiskLevel
0,high risk
1,high risk
2,high risk
3,high risk
4,low risk


## Step 1 — Data Loading

In [4]:
# Install the official UCI loader if needed, then import the dataset directly by UCI dataset ID.
# Run this once in a fresh environment if ucimlrepo is not installed:
# %pip install -U ucimlrepo

from ucimlrepo import fetch_ucirepo

maternal_health_risk = fetch_ucirepo(id=863)
X = maternal_health_risk.data.features.copy()
y = maternal_health_risk.data.targets.copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nX.head():")
display(X.head())
print("\ny.head():")
display(y.head())

Features shape: (1014, 6)
Target shape: (1014, 1)

X.head():


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
0,25,130,80,15.0,98.0,86
1,35,140,90,13.0,98.0,70
2,29,90,70,8.0,100.0,80
3,30,140,85,7.0,98.0,70
4,35,120,60,6.1,98.0,76



y.head():


,RiskLevel
0,high risk
1,high risk
2,high risk
3,high risk
4,low risk


## Step 2 — Data Cleaning

In [19]:
initial_records = len(X)

print("Missing/null values across features:")
print(X.isnull().sum())

combined = X.copy()
combined["__target__"] = y.iloc[:, 0].values
cleaned = combined.drop_duplicates().reset_index(drop=True)

duplicates_dropped = initial_records - len(cleaned)
X = cleaned.drop(columns="__target__")
y = cleaned["__target__"].to_frame(name=y.columns[0])

print(f"\nInitial record count: {initial_records}")
print(f"Duplicates dropped: {duplicates_dropped}")
print(f"Remaining cleaned records: {len(X)}")

Missing/null values across features:
Age            0
SystolicBP     0
DiastolicBP    0
BS             0
BodyTemp       0
HeartRate      0
dtype: int64

Initial record count: 1014
Duplicates dropped: 562
Remaining cleaned records: 452


## Step 3 — Data Splitting

In [20]:

%pip install -U scikit-learn

from sklearn.model_selection import train_test_split

# RiskLevel is a categorical classification target, not a rating/score,
# so the original three classes are retained: low risk, mid risk, high risk.
X_train, X_test, y_train, y_test = train_test_split(
    X, y.iloc[:, 0], test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape[0])
print("Testing set size:", X_test.shape[0])

Note: you may need to restart the kernel to use updated packages.
Training set size: 361
Testing set size: 91


You should consider upgrading via the 'c:\Users\Manav\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


## Step 4 — Model Training

In [21]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

model.fit(X_train, y_train)

print("KNN training completed successfully.")

KNN training completed successfully.


## Step 5 — Model Evaluation

In [22]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 68.13%

Confusion Matrix:
[[12  4  2]
 [ 0 45  2]
 [ 2 19  5]]

Classification Report:
              precision    recall  f1-score   support

   high risk       0.86      0.67      0.75        18
    low risk       0.66      0.96      0.78        47
    mid risk       0.56      0.19      0.29        26

    accuracy                           0.68        91
   macro avg       0.69      0.61      0.61        91
weighted avg       0.67      0.68      0.63        91



## Step 6 — Sample Prediction / Inference

In [ ]:
import pandas as pd

FEATURE_COLUMNS = list(X.columns)


def predict_maternal_risk(patient_data):
    """Predict risk for one new patient record supplied by an application or API."""
    missing_features = [column for column in FEATURE_COLUMNS if column not in patient_data]
    extra_features = [column for column in patient_data if column not in FEATURE_COLUMNS]

    if missing_features:
        raise ValueError(f"Missing required features: {missing_features}")
    if extra_features:
        raise ValueError(f"Unexpected features: {extra_features}")

    patient_frame = pd.DataFrame([patient_data], columns=FEATURE_COLUMNS)
    predicted_label = model.predict(patient_frame)[0]
    probabilities = model.predict_proba(patient_frame)[0]
    confidence = float(max(probabilities))

    return {
        "predicted_risk": predicted_label,
        "confidence": confidence,
    }


# Example of an unseen patient record arriving from a form, API request, or app.
new_patient = {
    "Age": 28,
    "SystolicBP": 118,
    "DiastolicBP": 76,
    "BS": 6.1,
    "BodyTemp": 98.4,
    "HeartRate": 82,
}

prediction = predict_maternal_risk(new_patient)
print("New patient input:")
display(pd.DataFrame([new_patient]))
print(f"Predicted risk: {prediction['predicted_risk']}")
print(f"Model confidence: {prediction['confidence']:.1%}")

Sample features:


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
299,35,100,60,15.0,98.0,80


Actual label: high risk
Predicted label: high risk
